# Module 13 Lab — Observability as Governance Evidence

Instrument a synthetic enterprise procurement agent and turn its trajectory into structured governance evidence.

In [ ]:
%pip install -q "pydantic>=2" pandas opentelemetry-api opentelemetry-sdk
print("Dependencies installed.")

In [ ]:
from pydantic import BaseModel, Field
from typing import Any, Literal, Optional
from datetime import datetime, timezone
import pandas as pd, hashlib, json, re, uuid
pd.set_option("display.max_colwidth",120)
def now(): return datetime.now(timezone.utc).isoformat()

## 1. Governance evidence schema

In [ ]:
class EvidenceEvent(BaseModel):
    trace_id:str; span_id:str; parent_id:Optional[str]=None
    event_type:str; timestamp:str=Field(default_factory=now)
    agent_id:Optional[str]=None; principal_id:Optional[str]=None
    purpose:Optional[str]=None; policy_version:Optional[str]=None
    risk_tier:Optional[str]=None; decision:Optional[str]=None
    attributes:dict[str,Any]={}

TRACE="trace-"+uuid.uuid4().hex
events=[]
def emit(event_type,parent_id=None,**kw):
    e=EvidenceEvent(trace_id=TRACE,span_id=uuid.uuid4().hex[:16],parent_id=parent_id,event_type=event_type,**kw)
    events.append(e); return e
root=emit("workflow",agent_id="procurement-agent:v13",principal_id="user:pseudo-42",purpose="approved procurement",risk_tier="HIGH")
root

## 2. Retrieval provenance

In [ ]:
retrieval=emit("retrieval",root.span_id,agent_id=root.agent_id,attributes={
 "query":"approved laptop vendors","source_ids":["kb:vendor-42:v7","policy:procurement:v12"],
 "trust":["MEDIUM","HIGH"],"content_stored":False})
retrieval

## 3. Policy decision as structured evidence

In [ ]:
policy=emit("policy_decision",root.span_id,agent_id=root.agent_id,policy_version="payment-policy:3.4",
 risk_tier="HIGH",decision="ESCALATE",attributes={"reason_codes":["HIGH_VALUE","NEW_VENDOR"],"risk_score":0.82})
policy

## 4. Bind approval to exact action

In [ ]:
action={"tool":"po.create","vendor_id":"V-42","amount":12000,"currency":"CAD"}
def digest(obj): return hashlib.sha256(json.dumps(obj,sort_keys=True,separators=(",",":")).encode()).hexdigest()
approval=emit("human_approval",policy.span_id,principal_id="manager:pseudo-7",
 attributes={"action_hash":digest(action),"role":"procurement_manager","expires":"2026-08-17T12:00:00Z"})
digest(action)==approval.attributes["action_hash"]

## 5. Delegation evidence

In [ ]:
delegation=emit("delegation",root.span_id,agent_id="manager-agent:v4",attributes={
 "delegate":"procurement-agent:v13","scope":["vendor.read","po.create"],"purpose":"approved procurement",
 "max_amount":15000,"parent_delegation":"human-session:123"})
delegation

## 6. Tool execution evidence

In [ ]:
tool=emit("tool_call",root.span_id,agent_id=root.agent_id,decision="ALLOW",attributes={
 "tool":"po.create","tool_version":"2.1","argument_hash":digest(action),
 "authorization_decision_id":"authz-8821","approval_span_id":approval.span_id})
outcome=emit("outcome",tool.span_id,agent_id=root.agent_id,attributes={
 "status":"SUCCESS","transaction_id":"PO-9001","verified":True,"reversible":True})
outcome

## 7. Reconstruct trajectory

In [ ]:
df=pd.DataFrame([e.model_dump() for e in events])
display(df[["event_type","span_id","parent_id","decision","policy_version","risk_tier"]])

## 8. Sensitive-data redaction

In [ ]:
PATTERNS=[(re.compile(r"\bsk-[A-Za-z0-9_-]+\b"),"[REDACTED_API_KEY]"),
(re.compile(r"\b[\w.+-]+@[\w.-]+\.\w+\b"),"[REDACTED_EMAIL]")]
def redact(v):
    if isinstance(v,str):
        for p,r in PATTERNS:v=p.sub(r,v)
    return v
[redact("contact buyer@example.com"),redact("token sk-exampleSECRET")]

## 9. Safe telemetry allowlist

In [ ]:
ALLOWED={"tool","tool_version","argument_hash","authorization_decision_id","approval_span_id","status","transaction_id","verified","reversible","reason_codes","risk_score"}
def sanitize_attributes(attrs): return {k:redact(v) for k,v in attrs.items() if k in ALLOWED}
sanitize_attributes({"tool":"po.create","password":"secret","status":"SUCCESS"})

## 10. Evidence completeness

In [ ]:
REQUIRED={
 "LOW":{"agent_id","principal_id","purpose"},
 "HIGH":{"agent_id","principal_id","purpose","policy_version","risk_tier"}
}
def completeness(event,tier="HIGH"):
    d=event.model_dump()
    req=REQUIRED[tier]
    present=sum(bool(d.get(x)) for x in req)
    return present/len(req)
completeness(root,"HIGH")

## 11. Workflow-level evidence checks

In [ ]:
def workflow_checks(events):
    types={e.event_type for e in events}
    return {
      "has_identity":any(e.principal_id for e in events),
      "has_policy_decision":"policy_decision" in types,
      "has_approval":"human_approval" in types,
      "has_tool":"tool_call" in types,
      "has_verified_outcome":any(e.event_type=="outcome" and e.attributes.get("verified") for e in events)
    }
workflow_checks(events)

## 12. Governance metrics

In [ ]:
runs=pd.DataFrame([
["t1","ALLOW","LOW",False,True,10],["t2","ESCALATE","HIGH",True,True,120],
["t3","DENY","HIGH",False,False,0],["t4","ALLOW","MEDIUM",False,True,40],
["t5","ESCALATE","HIGH",True,True,80]],columns=["trace","decision","risk","human_approval","task_success","cost"])
metrics={
 "task_success":runs.task_success.mean(),
 "autonomous_action_rate":((runs.decision=="ALLOW") & ~runs.human_approval).mean(),
 "escalation_rate":(runs.decision=="ESCALATE").mean(),
 "denial_rate":(runs.decision=="DENY").mean(),
 "cost_per_success":runs.cost.sum()/runs.task_success.sum()}
metrics

## 13. Near-miss evidence

In [ ]:
near_misses=pd.DataFrame([
["n1","unauthorized_tool","DENY","HIGH"],
["n2","new_vendor_high_value","ESCALATE","HIGH"],
["n3","egress_attempt","DENY","CRITICAL"],
["n4","delegation_scope","DENY","HIGH"]],columns=["id","signal","control","risk"])
display(near_misses)

## 14. Control effectiveness

In [ ]:
controls=pd.DataFrame([
["authz",100,18,1,2],["approval",40,12,2,0],["egress",25,8,1,0]],
columns=["control","evaluations","blocked","false_positive","bypass"])
controls["block_rate"]=controls.blocked/controls.evaluations
controls["bypass_rate"]=controls.bypass/controls.evaluations
display(controls)

## 15. Risk-aware sampling

In [ ]:
def retain_trace(risk,decision,security_anomaly=False):
    if security_anomaly or risk in {"HIGH","CRITICAL"} or decision in {"DENY","ESCALATE"}: return True
    # deterministic demo sample; production can use configured probabilistic sampling
    return False
[(r,d,retain_trace(r,d)) for r,d in [("LOW","ALLOW"),("HIGH","ALLOW"),("LOW","DENY")]]

## 16. Anomaly signals

In [ ]:
def anomaly_score(x):
    score=0
    score+=.25 if x.get("new_tool_sequence") else 0
    score+=.25 if x.get("delegation_depth",0)>3 else 0
    score+=.25 if x.get("denials",0)>=3 else 0
    score+=.25 if x.get("large_data_read") else 0
    return min(score,1)
anomaly_score({"new_tool_sequence":True,"delegation_depth":5,"denials":4})

## 17. Audit evidence package

In [ ]:
def audit_package(trace_events):
    return {
      "trace_id":trace_events[0].trace_id,
      "identity":next((e.principal_id for e in trace_events if e.principal_id),None),
      "purpose":next((e.purpose for e in trace_events if e.purpose),None),
      "policy_decisions":[e.model_dump() for e in trace_events if e.event_type=="policy_decision"],
      "approvals":[e.model_dump() for e in trace_events if e.event_type=="human_approval"],
      "actions":[e.model_dump() for e in trace_events if e.event_type=="tool_call"],
      "outcomes":[e.model_dump() for e in trace_events if e.event_type=="outcome"],
    }
pkg=audit_package(events)
pkg.keys()

## 18. Incident reconstruction

In [ ]:
def timeline(trace_events):
    return pd.DataFrame([{
      "time":e.timestamp,"type":e.event_type,"agent":e.agent_id,"principal":e.principal_id,
      "decision":e.decision,"details":e.attributes} for e in trace_events]).sort_values("time")
display(timeline(events))

## 19. OpenTelemetry instrumentation

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter

provider=TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)
tracer=trace.get_tracer("oneplusi.agent-governance")

with tracer.start_as_current_span("agent.workflow") as span:
    span.set_attribute("governance.agent.id","procurement-agent:v13")
    span.set_attribute("governance.risk.tier","HIGH")
    span.set_attribute("governance.policy.version","payment-policy:3.4")
    with tracer.start_as_current_span("policy.evaluate") as p:
        p.set_attribute("governance.decision","ESCALATE")
print("OTel demo emitted.")

## 20. OpenTelemetry GenAI conventions

For real model instrumentation, prefer the current OpenTelemetry GenAI semantic conventions for standardized GenAI attributes and keep organization-specific governance fields in a controlled namespace.

Examples from current conventions include concepts such as:

```text
gen_ai.operation.name
gen_ai.request.model
gen_ai.usage.input_tokens
gen_ai.usage.output_tokens
```

Treat prompt/message/tool content as sensitive opt-in telemetry.

## 21. OpenAI Agents SDK tracing pattern

The current Agents SDK provides built-in traces/spans for agent execution, generations, function tools, guardrails and handoffs, plus metadata and custom processors.

```python
# Illustrative shape; requires openai-agents and credentials.
from agents import Agent, Runner, trace

with trace(
    "procurement_workflow",
    group_id="business-process-123",
    metadata={
        "risk_tier": "HIGH",
        "policy_version": "payment-policy:3.4",
        "purpose": "approved_procurement",
    },
):
    # result = Runner.run_sync(agent, "...")
    pass
```

Use governance metadata deliberately; do not put secrets or unnecessary personal data in traces.

## 22. CI observability assertions

In [ ]:
checks=workflow_checks(events)
assert all(checks.values()),checks
assert digest(action)==approval.attributes["action_hash"]
assert outcome.attributes["verified"] is True
print("Governance evidence assertions passed.")

## 23. Exercises

1. Add model-generation spans and token/cost metrics.
2. Add RAG source IDs, trust labels and document hashes.
3. Build a three-agent delegation chain and reconstruct authority.
4. Introduce a policy denial and verify that no tool span follows it.
5. Add an approval mutation attack and detect the action-hash mismatch.
6. Add a memory write/read and trace provenance across two runs.
7. Implement pseudonymization for user/tenant identifiers.
8. Compare random sampling with risk-aware retention.
9. Create an evidence-completeness SLO by risk tier.
10. Add a near-miss dashboard.
11. Export OTel telemetry to an OTLP-compatible backend.
12. Instrument the same workflow with OpenAI Agents SDK tracing.
13. Compare Phoenix or LangSmith views with your vendor-neutral evidence schema.
14. Generate an auditor-facing evidence package that excludes raw prompts.
15. Simulate an incident and reconstruct the full trajectory.

## 24. Key takeaways

- Observability is not automatically governance evidence.
- Capture identity, purpose, authority, policy decision and outcome.
- Trace the complete trajectory.
- Make policy/approval events structured.
- Verify real outcomes, not only API success.
- Treat retrieval, memory and delegation as evidence surfaces.
- Measure near misses and control effectiveness.
- Use risk-aware retention.
- Minimize sensitive telemetry.
- Protect the observability system itself.
- Prefer portable schemas and OpenTelemetry interoperability.
- Feed evidence into evaluation, incidents, audit and policy improvement.